In [0]:

# Notebook: 03_silver_layer.py
# Purpose : Flatten FHIR JSON, deduplicate, standardise types

from pyspark.sql import SparkSession
import requests, uuid
from datetime import datetime
from importlib import reload

from pyspark.sql.functions import (
    col, explode_outer, coalesce, to_timestamp, row_number, current_timestamp
)
from pyspark.sql.window import Window
import sys


# Add repository root to Python path for module imports
repo_root = "/Workspace/Users/vyshnavi.thunuguntla@outlook.com/databricks-repo-FHIR"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import and reload to pick up latest changes
import audit_utils
reload(audit_utils)
from audit_utils import audit_start, audit_end

from config.config import BRONZE_DB, SILVER_DB

spark = SparkSession.builder.getOrCreate()
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")


# ── PATIENT ───────────────────────────────────────────────────────────────────
def silver_patient():
    df = spark.table(f"{BRONZE_DB}.patient")

    flat = (
        df.select(
            col("id").alias("patient_id"),
            col("gender"),
            col("birthDate").alias("birth_date"),
            # col("deceasedBoolean").alias("is_deceased"),  # Removed - field not in data
            explode_outer("name").alias("name_struct"),
            col("_extraction_timestamp"),
            col("_ingestion_date"),
            col("load_timestamp"),
        )
        .withColumn("family_name",  col("name_struct.family"))
        .withColumn("given_name",   explode_outer(col("name_struct.given")))
        .drop("name_struct")
        .withColumn("birth_date",   to_timestamp("birth_date"))
    )

    # Deduplicate: keep latest record per patient_id
    w   = Window.partitionBy("patient_id").orderBy(col("load_timestamp").desc())
    flat = flat.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

    flat.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER_DB}.patient")
    print(f"[SILVER] patient: {flat.count()} rows")


# ── ENCOUNTER ─────────────────────────────────────────────────────────────────
def silver_encounter():
    df = spark.table(f"{BRONZE_DB}.encounter")

    flat = (
        df.select(
            col("id").alias("encounter_id"),
            col("status"),
            col("class.code").alias("encounter_class"),
            col("subject.reference").alias("patient_ref"),
            col("period.start").alias("period_start"),
            col("period.end").alias("period_end"),
            col("_extraction_timestamp"),
            col("_ingestion_date"),
            col("load_timestamp"),
        )
        .withColumn("period_start", to_timestamp("period_start"))
        .withColumn("period_end",   to_timestamp("period_end"))
        .withColumn("patient_id",   col("patient_ref").substr(9, 100))  # strip "Patient/"
    )

    w    = Window.partitionBy("encounter_id").orderBy(col("load_timestamp").desc())
    flat = flat.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

    flat.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER_DB}.encounter")
    print(f"[SILVER] encounter: {flat.count()} rows")


# ── OBSERVATION ───────────────────────────────────────────────────────────────
def silver_observation():
    df = spark.table(f"{BRONZE_DB}.observation")

    flat = (
        df.select(
            col("id").alias("observation_id"),
            col("status"),
            col("subject.reference").alias("patient_ref"),
            col("encounter.reference").alias("encounter_ref"),
            col("effectiveDateTime").alias("effective_datetime"),
            col("code.coding").getItem(0).getField("code").alias("loinc_code"),
            col("code.coding").getItem(0).getField("display").alias("observation_name"),
            col("valueQuantity.value").alias("value_quantity"),
            col("valueQuantity.unit").alias("value_unit"),
            col("_extraction_timestamp"),
            col("_ingestion_date"),
            col("load_timestamp"),
        )
        .withColumn("effective_datetime", to_timestamp("effective_datetime"))
        .withColumn("patient_id",   col("patient_ref").substr(9, 100))
        .withColumn("encounter_id", col("encounter_ref").substr(11, 100))
    )

    w    = Window.partitionBy("observation_id").orderBy(col("load_timestamp").desc())
    flat = flat.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

    flat.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER_DB}.observation")
    print(f"[SILVER] observation: {flat.count()} rows")


# ── CONDITION ─────────────────────────────────────────────────────────────────
def silver_condition():
    df = spark.table(f"{BRONZE_DB}.condition")

    flat = (
        df.select(
            col("id").alias("condition_id"),
            col("clinicalStatus.coding").getItem(0).getField("code").alias("clinical_status"),
            col("subject.reference").alias("patient_ref"),
            col("encounter.reference").alias("encounter_ref"),
            col("code.coding").getItem(0).getField("code").alias("snomed_code"),
            col("code.coding").getItem(0).getField("display").alias("condition_name"),
            col("onsetDateTime").alias("onset_datetime"),
            col("_extraction_timestamp"),
            col("_ingestion_date"),
            col("load_timestamp"),
        )
        .withColumn("onset_datetime", to_timestamp("onset_datetime"))
        .withColumn("patient_id",   col("patient_ref").substr(9, 100))
        .withColumn("encounter_id", col("encounter_ref").substr(11, 100))
    )

    w    = Window.partitionBy("condition_id").orderBy(col("load_timestamp").desc())
    flat = flat.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

    flat.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER_DB}.condition")
    print(f"[SILVER] condition: {flat.count()} rows")

run_id = str(uuid.uuid4()) 
patient_table_name = f"{SILVER_DB}.patient"
CNT1 = spark.sql(f"SELECT COUNT(*) FROM {patient_table_name} ").collect()[0][0]
started_at = datetime.utcnow() # current_timestamp(),
audit_start(
        run_id        = run_id,
        stage         = "SILVER",
        resource_type = 'patient',
        operation     = "INGEST",
        status        = "STARTED",
        started_at = started_at ,
        before_run_source_count = CNT1 ,
        created_by    = "02_silver_ingest"
    )
silver_patient()
CNT2 = spark.sql(f"SELECT COUNT(*) FROM {patient_table_name} ").collect()[0][0]
audit_end(
        run_id        = run_id,
        stage         = "SILVER",
        resource_type = 'patient',
        operation     = "INGEST",
        status        = "ENDED",
        post_run_source_count = CNT2 ,
    )
#####
run_id = str(uuid.uuid4()) 
encounter_table_name = f"{SILVER_DB}.encounter"
CNT1 = spark.sql(f"SELECT COUNT(*) FROM {encounter_table_name} ").collect()[0][0]
started_at = datetime.utcnow() # current_timestamp(),
audit_start(
        run_id        = run_id,
        stage         = "SILVER",
        resource_type = 'encounter',
        operation     = "INGEST",
        status        = "STARTED",
        started_at = started_at ,
        before_run_source_count = CNT1 ,
        created_by    = "02_silver_ingest"
    )

silver_encounter()
CNT2 = spark.sql(f"SELECT COUNT(*) FROM {encounter_table_name} ").collect()[0][0]
audit_end(
        run_id        = run_id,
        stage         = "SILVER",
        resource_type = 'encounter',
        operation     = "INGEST",
        status        = "ENDED",
        post_run_source_count = CNT2 ,
    )
#######
run_id = str(uuid.uuid4()) 
observation_table_name = f"{SILVER_DB}.observation"
CNT1 = spark.sql(f"SELECT COUNT(*) FROM {observation_table_name} ").collect()[0][0]
started_at = datetime.utcnow() # current_timestamp(),
audit_start(
        run_id        = run_id,
        stage         = "SILVER",
        resource_type = 'observation',
        operation     = "INGEST",
        status        = "STARTED",
        started_at = started_at ,
        before_run_source_count = CNT1 ,
        created_by    = "02_silver_ingest"
    )

silver_observation()
CNT2 = spark.sql(f"SELECT COUNT(*) FROM {observation_table_name} ").collect()[0][0]
audit_end(
        run_id        = run_id,
        stage         = "SILVER",
        resource_type = 'observation',
        operation     = "INGEST",
        status        = "ENDED",
        post_run_source_count = CNT2 ,
    )

run_id = str(uuid.uuid4()) 
condition_table_name = f"{SILVER_DB}.condition"
CNT1 = spark.sql(f"SELECT COUNT(*) FROM {condition_table_name} ").collect()[0][0]
started_at = datetime.utcnow() # current_timestamp(),
########
audit_start(
        run_id        = run_id,
        stage         = "SILVER",
        resource_type = 'condition',
        operation     = "INGEST",
        status        = "STARTED",
        started_at = started_at ,
        before_run_source_count = CNT1 ,
        created_by    = "02_silver_ingest"
    )

silver_condition()
CNT2 = spark.sql(f"SELECT COUNT(*) FROM {condition_table_name} ").collect()[0][0]
audit_end(
        run_id        = run_id,
        stage         = "SILVER",
        resource_type = 'condition',
        operation     = "INGEST",
        status        = "ENDED",
        post_run_source_count = CNT2 ,
    )

#silver_patient()
#silver_encounter()
#silver_observation()
#silver_condition()
print("✅ Silver layer complete.")